# Returns analysis

Handed over by a placement student who left in July. The last thing they wrote in
the handover email was that the model gets 0.94 and is ready to show the client.

Nobody has run this since.

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv('../data/retail_orders_week1.csv')
df.shape

(2030, 17)

In [4]:
# drop the rows that look wrong, and put VAT on the order value
df = df[df['quantity'] > 0]
df = df[df['delivery_days'] > 0]
df = df.assign(order_value_incl_vat=df['order_value'] * 1.2)
len(df)

2021

In [5]:
sample = df.sample(1200, random_state=7)
sample['returned'].mean()

np.float64(0.1975)

In [6]:
features = ['export_seq', 'item_price', 'discount_pct', 'quantity',
            'delivery_days', 'customer_prior_orders']

X = df[features]
y = df['returned']

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.25, random_state=7, stratify=y)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print('ROC-AUC', round(auc, 3))

ROC-AUC 0.938


In [8]:
coefs = pd.Series(model.coef_[0], index=features).sort_values()
coefs

export_seq              -3.214779
customer_prior_orders   -0.372909
quantity                 0.044697
item_price               0.056841
discount_pct             0.069166
delivery_days            0.072927
dtype: float64

In [9]:
summary = (df.groupby('category')['returned']
.mean()
.sort_values(ascending=False)
.rename('return_rate')
.reset_index())
summary

,category,return_rate
0,footwear,0.292683
1,apparel,0.238462
2,electronics,0.203762
3,sports,0.129412
4,home,0.111959
5,beauty,0.079208


0.94 is good enough. Send to client Friday.

In [10]:
df.groupby('returned')['export_seq'].describe()[['min','max','mean']]

,min,max,mean
returned,,,
0,1.0,2030.0,1185.410398
1,7.0,600.0,306.396373
